# Redrob Hackathon - Team Antigravity Sandbox
This notebook executes the exact offline/runtime pipeline on a 100-candidate sample.

In [ ]:
import os
# Prevent nested cloning if the cell is run multiple times
if os.path.exists("/content/Hack2Skill26"):
    !rm -rf /content/Hack2Skill26

%cd /content
!git clone -b sandbox https://github.com/ADITYASINGH1206/Hack2Skill26.git
%cd /content/Hack2Skill26
!pip install -r requirements.txt -q

### (Optional) Upload Custom Candidate Sample
Run this cell to upload your own `.jsonl` sample file (max 100 candidates). If you skip this, it will automatically use our pre-loaded `sample_candidates.jsonl` from the repository.

In [ ]:
from google.colab import files

INPUT_FILE = "sample_candidates.jsonl"

print("Upload a .jsonl file, or cancel to use the default sample:")
try:
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        INPUT_FILE = filename
        print(f"\n[*] Using uploaded file: {INPUT_FILE}")
    else:
        print(f"\n[*] No file uploaded. Defaulting to pre-loaded {INPUT_FILE}")
except Exception:
    print(f"\n[*] Upload skipped or failed. Defaulting to pre-loaded {INPUT_FILE}")

### Phase 1: Data Engineering & Pre-computation
Execute the offline pipeline to clean data and generate FAISS/BM25 artifacts in the `artifacts/` folder.

In [ ]:
# 0. Wipe existing full-scale artifacts from the repo to guarantee small-sample reproduction
!rm -rf artifacts/*

# 1. Clean the sample candidates (Drops honeypots & traps)
!python pipeline/ingest_and_filter.py {INPUT_FILE} clean_pool.jsonl

# 2. Generate FAISS and BM25 artifacts into the /content/Hack2Skill26/artifacts directory
!python pipeline/indexer.py clean_pool.jsonl

### Phase 2: Runtime Inference Engine
Execute the 5-minute timed ranking script on the generated artifacts.

In [ ]:
!python rank.py --out sandbox_output.csv

### Output Validation
Verify the CSV structure and ranking results.

In [ ]:
import pandas as pd
try:
    df = pd.read_csv("sandbox_output.csv")
    print(f"\nTotal Ranked: {len(df)}")
    display(df.head(10))
except FileNotFoundError:
    print("[!] Error: Output file not found. Did the pipeline complete successfully?")